In [1]:
import pandas as pd

df = pd.read_excel(
    r"C:\& Satheesh\Campaign-Conversion-Prediction-Model\data\Financial_Services_KPI_Dashboard_5000Rows.xlsx"
)

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df.head()

,customer_id,customer_name,product_type,fresco_segment,age_band,region,marketing_channel,campaign_date,days_to_maturity,account_value,...,online_login_days,complaints,email_open_rate,web_visits_30d,previous_campaign_response,conversion_flag,conversion_rate,campaign_cost,campaign_roi,estimated_revenue
0,100001,Nathan Gilbert DVM,CTF,Mature Wealth,60+,Glasgow,Email,2024-10-10,99,39771.22,...,125,2,77.91,12,Yes,1,9.34,452.44,1.41,2137.92
1,100002,Andrew Kelley,CTF,Mature Wealth,36-45,Manchester,SMS,2025-09-30,59,41230.72,...,157,2,50.13,15,No,0,15.31,223.29,1.18,2734.62
2,100003,Chelsea Lowery,ISA,Digital Achievers,60+,Liverpool,Branch,2023-10-05,151,51266.82,...,152,4,37.40,17,No,1,11.14,154.93,7.89,1359.88
3,100004,Christopher Barry,ISA,Mature Wealth,60+,Leeds,Digital Campaign,2024-05-03,27,73215.67,...,44,4,82.79,30,No,1,16.95,353.77,6.02,5770.11
4,100005,Tony Lopez MD,CTF,Mature Wealth,18-25,Liverpool,SMS,2025-05-13,199,55519.51,...,31,4,55.93,21,No,1,2.99,46.19,1.09,649.18


In [3]:
df["conversion_flag"] = (
    (
        (df["email_open_rate"] >= 40) &
        (df["web_visits_30d"] >= 5) &
        (df["previous_campaign_response"] == "Yes")
    )
    |
    (
        (df["active_dd"] == "Yes") &
        (df["failed_dd"] == "No") &
        (df["account_value"] >= 15000)
    )
).astype(int)

df["conversion_flag"].value_counts()

conversion_flag
0    2969
1    2031
Name: count, dtype: int64

In [5]:
features = [
    "product_type",
    "fresco_segment",
    "age_band",
    "region",
    "marketing_channel",
    "days_to_maturity",
    "account_value",
    "monthly_contribution",
    "active_dd",
    "failed_dd",
    "online_login_days",
    "complaints",
    "email_open_rate",
    "web_visits_30d",
    "previous_campaign_response",
    "campaign_cost"
]

target = "conversion_flag"

X = df[features]
y = df[target]

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

categorical_features = [
    "product_type",
    "fresco_segment",
    "age_band",
    "region",
    "marketing_channel",
    "active_dd",
    "failed_dd",
    "previous_campaign_response"
]

numeric_features = [
    "days_to_maturity",
    "account_value",
    "monthly_contribution",
    "online_login_days",
    "complaints",
    "email_open_rate",
    "web_visits_30d",
    "campaign_cost"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=10000))
    ]
)

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['days_to_maturity',
                                                   'account_value',
                                                   'monthly_contribution',
                                                   'online_login_days',
                                                   'complaints',
                                                   'email_open_rate',
                                                   'web_visits_30d',
                                                   'campaign_cost']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['product_type',
                                                   'fresco_segment', 'age_band',
                                                   'region',
                                                   'marketing_channel',
                                                   'active_dd', 'failed_dd',
                                                   'previous_campaign_response'])])),
                ('classifier', LogisticRegression(max_iter=10000))])

In [11]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.83

Confusion Matrix:
[[516  78]
 [ 92 314]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       594
           1       0.80      0.77      0.79       406

    accuracy                           0.83      1000
   macro avg       0.82      0.82      0.82      1000
weighted avg       0.83      0.83      0.83      1000



In [13]:
y_probability = model.predict_proba(X_test)[:, 1]

results = X_test.copy()
results["actual_conversion"] = y_test.values
results["predicted_conversion"] = y_pred
results["conversion_probability"] = y_probability

results.head()

,product_type,fresco_segment,age_band,region,marketing_channel,days_to_maturity,account_value,monthly_contribution,active_dd,failed_dd,online_login_days,complaints,email_open_rate,web_visits_30d,previous_campaign_response,campaign_cost,actual_conversion,predicted_conversion,conversion_probability
1120,ISA,Budget Households,60+,Bristol,Email,287,27079.81,893.22,Yes,No,138,3,29.64,15,No,331.93,1,0,0.270082
2588,ISA,Budget Households,36-45,Manchester,SMS,346,51870.63,1082.55,No,No,115,2,30.57,10,Yes,455.99,0,1,0.578126
1533,CTF,Retired Comfort,60+,Birmingham,Branch,105,74660.84,1001.01,No,No,67,4,41.73,6,Yes,285.28,1,1,0.716787
2859,CTF,Affluent Families,18-25,Manchester,Email,285,13901.16,436.88,Yes,Yes,104,1,21.48,7,Yes,181.20,0,0,0.146372
1407,LISA,Budget Households,36-45,Leeds,SMS,164,39222.38,256.75,Yes,Yes,41,0,75.90,25,Yes,53.58,1,1,0.953161


In [15]:
def conversion_priority(probability):
    if probability >= 0.70:
        return "High Conversion Potential"
    elif probability >= 0.40:
        return "Medium Conversion Potential"
    else:
        return "Low Conversion Potential"

results["conversion_priority"] = results["conversion_probability"].apply(conversion_priority)

results.head()


,product_type,fresco_segment,age_band,region,marketing_channel,days_to_maturity,account_value,monthly_contribution,active_dd,failed_dd,online_login_days,complaints,email_open_rate,web_visits_30d,previous_campaign_response,campaign_cost,actual_conversion,predicted_conversion,conversion_probability,conversion_priority
1120,ISA,Budget Households,60+,Bristol,Email,287,27079.81,893.22,Yes,No,138,3,29.64,15,No,331.93,1,0,0.270082,Low Conversion Potential
2588,ISA,Budget Households,36-45,Manchester,SMS,346,51870.63,1082.55,No,No,115,2,30.57,10,Yes,455.99,0,1,0.578126,Medium Conversion Potential
1533,CTF,Retired Comfort,60+,Birmingham,Branch,105,74660.84,1001.01,No,No,67,4,41.73,6,Yes,285.28,1,1,0.716787,High Conversion Potential
2859,CTF,Affluent Families,18-25,Manchester,Email,285,13901.16,436.88,Yes,Yes,104,1,21.48,7,Yes,181.20,0,0,0.146372,Low Conversion Potential
1407,LISA,Budget Households,36-45,Leeds,SMS,164,39222.38,256.75,Yes,Yes,41,0,75.90,25,Yes,53.58,1,1,0.953161,High Conversion Potential


In [17]:
output_path = r"C:\& Satheesh\Campaign-Conversion-Prediction-Model\output\campaign_conversion_predictions.xlsx"

results.to_excel(output_path, index=False)

print("Prediction output exported successfully.")

Prediction output exported successfully.


## Saving Trained Model 

In [21]:
import joblib

# Save trained model
joblib.dump(model, "conversion_model.pkl")

print("Model saved successfully.")

Model saved successfully.


## Loading Trained Model

In [23]:
import joblib

# Load trained model
loaded_model = joblib.load("conversion_model.pkl")

print("Model loaded successfully.")

Model loaded successfully.


## Reading Future Customer Dataset

In [25]:
future_customers = pd.read_excel(
    r"C:\& Satheesh\Campaign-Conversion-Prediction-Model\data\future_customers_1000.xlsx"
)

future_customers.columns = future_customers.columns.str.strip().str.lower().str.replace(" ", "_")

future_customers.head()

,customer_id,customer_name,product_type,fresco_segment,age_band,region,marketing_channel,campaign_date,days_to_maturity,account_value,...,failed_dd,online_login_days,complaints,email_open_rate,web_visits_30d,previous_campaign_response,conversion_rate,campaign_cost,campaign_roi,estimated_revenue
0,104739,Charles Brewer,JISA,Retired Comfort,46-60,London,SMS,2023-11-12,281,29997.06,...,Yes,72,3,66.09,9,Yes,11.67,373.24,7.45,1149.60
1,104491,Tracy Pruitt,LISA,Budget Households,46-60,Bristol,Branch,2024-06-03,50,8214.80,...,No,103,2,58.59,27,Yes,14.47,321.19,5.20,389.17
2,100107,Tracy Williams,CTF,Retired Comfort,46-60,Liverpool,Digital Campaign,2024-06-11,149,10585.87,...,No,28,2,37.03,4,No,6.33,42.32,2.80,590.49
3,101538,Ryan Davis,CTF,Affluent Families,36-45,Liverpool,Branch,2025-01-12,164,38982.05,...,Yes,152,1,73.78,30,No,9.49,410.57,5.82,2491.43
4,104848,Melissa Harris,CTF,Digital Achievers,60+,London,Email,2024-09-19,235,18852.31,...,Yes,121,4,24.49,12,No,6.91,228.51,4.53,1467.40


In [27]:
future_X = future_customers[features]

In [29]:
loaded_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['days_to_maturity',
                                                   'account_value',
                                                   'monthly_contribution',
                                                   'online_login_days',
                                                   'complaints',
                                                   'email_open_rate',
                                                   'web_visits_30d',
                                                   'campaign_cost']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['product_type',
                                                   'fresco_segment', 'age_band',
                                                   'region',
                                                   'marketing_channel',
                                                   'active_dd', 'failed_dd',
                                                   'previous_campaign_response'])])),
                ('classifier', LogisticRegression(max_iter=10000))])

In [31]:
future_predictions = loaded_model.predict(future_X)

future_probabilities = loaded_model.predict_proba(future_X)[:, 1]

In [33]:
future_results = future_customers.copy()

future_results["predicted_conversion"] = future_predictions

future_results["conversion_probability"] = future_probabilities

future_results.head()

,customer_id,customer_name,product_type,fresco_segment,age_band,region,marketing_channel,campaign_date,days_to_maturity,account_value,...,complaints,email_open_rate,web_visits_30d,previous_campaign_response,conversion_rate,campaign_cost,campaign_roi,estimated_revenue,predicted_conversion,conversion_probability
0,104739,Charles Brewer,JISA,Retired Comfort,46-60,London,SMS,2023-11-12,281,29997.06,...,3,66.09,9,Yes,11.67,373.24,7.45,1149.60,0,0.203460
1,104491,Tracy Pruitt,LISA,Budget Households,46-60,Bristol,Branch,2024-06-03,50,8214.80,...,2,58.59,27,Yes,14.47,321.19,5.20,389.17,1,0.804754
2,100107,Tracy Williams,CTF,Retired Comfort,46-60,Liverpool,Digital Campaign,2024-06-11,149,10585.87,...,2,37.03,4,No,6.33,42.32,2.80,590.49,0,0.028100
3,101538,Ryan Davis,CTF,Affluent Families,36-45,Liverpool,Branch,2025-01-12,164,38982.05,...,1,73.78,30,No,9.49,410.57,5.82,2491.43,0,0.065448
4,104848,Melissa Harris,CTF,Digital Achievers,60+,London,Email,2024-09-19,235,18852.31,...,4,24.49,12,No,6.91,228.51,4.53,1467.40,0,0.012926


In [35]:
future_results["conversion_priority"] = future_results[
    "conversion_probability"
].apply(conversion_priority)

future_results.head()

,customer_id,customer_name,product_type,fresco_segment,age_band,region,marketing_channel,campaign_date,days_to_maturity,account_value,...,email_open_rate,web_visits_30d,previous_campaign_response,conversion_rate,campaign_cost,campaign_roi,estimated_revenue,predicted_conversion,conversion_probability,conversion_priority
0,104739,Charles Brewer,JISA,Retired Comfort,46-60,London,SMS,2023-11-12,281,29997.06,...,66.09,9,Yes,11.67,373.24,7.45,1149.60,0,0.203460,Low Conversion Potential
1,104491,Tracy Pruitt,LISA,Budget Households,46-60,Bristol,Branch,2024-06-03,50,8214.80,...,58.59,27,Yes,14.47,321.19,5.20,389.17,1,0.804754,High Conversion Potential
2,100107,Tracy Williams,CTF,Retired Comfort,46-60,Liverpool,Digital Campaign,2024-06-11,149,10585.87,...,37.03,4,No,6.33,42.32,2.80,590.49,0,0.028100,Low Conversion Potential
3,101538,Ryan Davis,CTF,Affluent Families,36-45,Liverpool,Branch,2025-01-12,164,38982.05,...,73.78,30,No,9.49,410.57,5.82,2491.43,0,0.065448,Low Conversion Potential
4,104848,Melissa Harris,CTF,Digital Achievers,60+,London,Email,2024-09-19,235,18852.31,...,24.49,12,No,6.91,228.51,4.53,1467.40,0,0.012926,Low Conversion Potential


In [37]:
future_output_path = r"C:\& Satheesh\Campaign-Conversion-Prediction-Model\output\future_customer_predictions.xlsx"

future_results.to_excel(future_output_path, index=False)

print("Future customer predictions exported successfully.")

Future customer predictions exported successfully.
